# Level 3 — Numerical Methods

## Learning Objectives
1. **Root Finding**: Bisection, Newton-Raphson, Secant methods for solving f(x)=0
2. **Numerical Integration**: Trapezoidal and Simpson's rule for computing integrals
3. **Linear Solvers**: Gaussian elimination for solving Ax=b (water allocation)
4. **Application**: Optimize irrigation timing and volume using root-finding

## Why This Matters
- **Root finding**: Determine optimal irrigation volume to maintain target soil moisture
- **Integration**: Compute cumulative ET, cumulative demand over season
- **Linear solvers**: Water allocation across multiple zones with constraints
- **Practical**: Real-time optimization for irrigation decision support

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load data
repo_root = Path('.').resolve().parent
data_dir = repo_root / 'data' / 'raw'
weather = pd.read_csv(data_dir / 'weather_daily.csv')
crops = pd.read_csv(data_dir / 'crop_zone_parameters.csv')

print(f"Loaded {len(weather)} weather records")
print(f"Crop zones: {list(crops['Zone'].unique())}")

In [ ]:
# ROOT FINDING: Bisection Method
def bisection(f, a, b, tol=1e-6, max_iter=100):
    """
    Bisection method to find root of f(x) = 0 in interval [a, b].
    
    Parameters:
    -----------
    f : callable
        Function to find root of
    a, b : float
        Initial bracket (must have f(a)*f(b) < 0)
    tol : float
        Tolerance for convergence
    max_iter : int
        Maximum iterations
    
    Returns:
    --------
    root : float
        Approximate root
    iterations : int
        Number of iterations used
    """
    if f(a) * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs")
    
    for it in range(max_iter):
        c = (a + b) / 2
        if abs(b - a) < tol:
            return c, it
        
        if f(c) == 0:
            return c, it
        elif f(a) * f(c) < 0:
            b = c
        else:
            a = c
    
    return (a + b) / 2, max_iter

# ROOT FINDING: Newton-Raphson Method
def newton_raphson(f, df, x0, tol=1e-6, max_iter=100):
    """
    Newton-Raphson method: x_{n+1} = x_n - f(x_n)/f'(x_n)
    
    Parameters:
    -----------
    f : callable
        Function to find root of
    df : callable
        Derivative of f
    x0 : float
        Initial guess
    tol : float
        Tolerance for convergence
    max_iter : int
        Maximum iterations
    
    Returns:
    --------
    root : float
        Approximate root
    iterations : int
        Number of iterations used
    """
    x = x0
    for it in range(max_iter):
        fx = f(x)
        if abs(fx) < tol:
            return x, it
        
        dfx = df(x)
        if abs(dfx) < 1e-12:
            raise ValueError("Derivative too small")
        
        x_new = x - fx / dfx
        if abs(x_new - x) < tol:
            return x_new, it
        x = x_new
    
    return x, max_iter

# ROOT FINDING: Secant Method
def secant(f, x0, x1, tol=1e-6, max_iter=100):
    """
    Secant method: like Newton-Raphson but approximates derivative.
    Useful when derivative is expensive to compute.
    
    x_{n+1} = x_n - f(x_n) * (x_n - x_{n-1}) / (f(x_n) - f(x_{n-1}))
    """
    for it in range(max_iter):
        f0 = f(x0)
        f1 = f(x1)
        
        if abs(f1 - f0) < 1e-12:
            return x1, it
        
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        
        if abs(x2 - x1) < tol:
            return x2, it
        
        x0, x1 = x1, x2
    
    return x1, max_iter

print("Root-finding methods defined.")

In [ ]:
# APPLICATION: Find optimal irrigation volume
# Problem: Given soil moisture S(t), target T, what irrigation I makes S(t+1) = T?
# Water balance: S(t+1) = S(t) + P - ET - D + I
# Solve for I: I = T - S(t) - P + ET + D

def optimal_irrigation(s_current, target_moisture, rainfall, et, drainage=1.0):
    """
    Find irrigation needed to reach target soil moisture tomorrow.
    
    S(t+1) = S(t) + P - ET - D + I = Target
    => I = Target - S(t) - P + ET + D
    
    But if I < 0, no irrigation needed (soil has enough water).
    """
    i_needed = target_moisture - s_current - rainfall + et + drainage
    return max(i_needed, 0.0)  # Can't irrigate negative amount

# Test case
s = 120  # Current soil moisture
target = 150  # Target for next day
p = 2  # Expected rainfall
et = 4  # Expected ET
d = 1  # Drainage

i_opt = optimal_irrigation(s, target, p, et, d)
print(f"\nCurrent soil moisture: {s} mm")
print(f"Expected rainfall: {p} mm")
print(f"Expected ET: {et} mm")
print(f"Expected drainage: {d} mm")
print(f"Target for next day: {target} mm")
print(f"Optimal irrigation: {i_opt:.2f} mm")

# Verify
s_next = s + p - et - d + i_opt
print(f"\nVerification: S(t+1) = {s_next:.2f} mm (target: {target} mm)")

In [ ]:
# NUMERICAL INTEGRATION: Trapezoidal Rule
def trapezoidal_rule(y, x=None, dx=1.0):
    """
    Trapezoidal rule for numerical integration.
    Approximates integral as sum of trapezoids.
    
    Parameters:
    -----------
    y : array
        Function values at sample points
    x : array, optional
        x-coordinates (if not uniform)
    dx : float, optional
        Uniform spacing (if x not provided)
    
    Returns:
    --------
    integral : float
        Approximate integral
    """
    y = np.asarray(y)
    if x is not None:
        x = np.asarray(x)
        dx = np.diff(x)
        return np.sum((y[:-1] + y[1:]) / 2 * dx)
    else:
        return dx * (np.sum(y[:-1] + y[1:]) / 2)

# NUMERICAL INTEGRATION: Simpson's Rule
def simpsons_rule(y, x=None, dx=1.0):
    """
    Simpson's rule: more accurate than trapezoidal.
    Requires even number of intervals.
    """
    y = np.asarray(y)
    n = len(y) - 1
    
    if n % 2 != 0:
        # If odd, use last interval as trapezoid
        return simpsons_rule(y[:-1], x=x, dx=dx) + (y[-2] + y[-1]) / 2 * dx
    
    if x is not None:
        x = np.asarray(x)
        dx = x[1] - x[0]  # Assume uniform
    
    integral = y[0] + y[-1]
    integral += 4 * np.sum(y[1:-1:2])
    integral += 2 * np.sum(y[2:-1:2])
    integral *= dx / 3
    return integral

# Test: Integrate sin(x) from 0 to pi
x_test = np.linspace(0, np.pi, 101)
y_test = np.sin(x_test)

integral_trap = trapezoidal_rule(y_test, x=x_test)
integral_simp = simpsons_rule(y_test, x=x_test)
integral_exact = 2.0  # Analytical integral of sin(x) from 0 to pi

print("\n=== NUMERICAL INTEGRATION TEST ===")
print(f"Trapezoidal:  {integral_trap:.6f}")
print(f"Simpson's:    {integral_simp:.6f}")
print(f"Exact (2π):   {integral_exact:.6f}")
print(f"Error (trap): {abs(integral_trap - integral_exact):.2e}")
print(f"Error (simp): {abs(integral_simp - integral_exact):.2e}")

In [ ]:
# APPLICATION: Cumulative ET demand over season
# Integrate ET curve to find total water demand

et_seasonal = []
for i in range(len(weather)):
    kc = 0.8  # Crop coefficient
    t_mean = (weather.iloc[i]['Tmax_C'] + weather.iloc[i]['Tmin_C']) / 2
    t_range = max(weather.iloc[i]['Tmax_C'] - weather.iloc[i]['Tmin_C'], 0.1)
    rad = weather.iloc[i]['SolarRadiation_MJm2']
    hum = weather.iloc[i]['Humidity_%']
    
    et = 0.0023 * rad * (t_mean + 17.8) * np.sqrt(t_range) * kc
    humidity_factor = 1.0 - (hum - 50) / 100 if hum > 50 else 1.0
    et = et * max(humidity_factor, 0.5)
    et_seasonal.append(max(et, 0.0))

et_seasonal = np.array(et_seasonal)
days = np.arange(len(et_seasonal))

# Cumulative ET
cumulative_et = np.cumsum(et_seasonal)

# Integrate using trapezoidal rule
total_et_trap = trapezoidal_rule(et_seasonal, dx=1.0)
total_et_simp = simpsons_rule(et_seasonal, dx=1.0)

print(f"\nTotal ET (trapezoidal): {total_et_trap:.2f} mm")
print(f"Total ET (Simpson's):   {total_et_simp:.2f} mm")
print(f"Average daily ET:       {total_et_trap / len(et_seasonal):.2f} mm/day")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Daily ET
ax1.bar(days, et_seasonal, color='orange', alpha=0.7)
ax1.set_xlabel('Day of Season')
ax1.set_ylabel('ET (mm/day)')
ax1.set_title('Daily Evapotranspiration')
ax1.grid(True, alpha=0.3)

# Cumulative ET
ax2.plot(days, cumulative_et, linewidth=2, color='darkgreen')
ax2.fill_between(days, cumulative_et, alpha=0.3, color='green')
ax2.set_xlabel('Day of Season')
ax2.set_ylabel('Cumulative ET (mm)')
ax2.set_title('Cumulative Evapotranspiration')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# LINEAR SOLVER: Gaussian Elimination (Ax = b)
def gaussian_elimination(A, b):
    """
    Solve linear system Ax = b using Gaussian elimination with partial pivoting.
    
    Parameters:
    -----------
    A : ndarray, shape (n, n)
        Coefficient matrix
    b : ndarray, shape (n,)
        Right-hand side
    
    Returns:
    --------
    x : ndarray
        Solution vector
    """
    n = len(b)
    A = A.astype(float)
    b = b.astype(float)
    
    # Forward elimination with partial pivoting
    for i in range(n):
        # Find pivot
        max_row = i + np.argmax(np.abs(A[i:, i]))
        A[[i, max_row]] = A[[max_row, i]]
        b[[i, max_row]] = b[[max_row, i]]
        
        # Eliminate column
        for j in range(i+1, n):
            factor = A[j, i] / A[i, i]
            A[j, i:] -= factor * A[i, i:]
            b[j] -= factor * b[i]
    
    # Back substitution
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    
    return x

# APPLICATION: Water allocation problem
# 3 irrigation zones (A, B, C) with constraints
# Zone A: 20 ha, needs 1200 mm, cost 5 units/mm
# Zone B: 30 ha, needs 1400 mm, cost 4 units/mm
# Zone C: 15 ha, needs 1000 mm, cost 6 units/mm
# Total water available: 110,000 mm

# Linear system: allocate water to minimize cost
# Constraint: sum of allocations = 110,000
# But simplified here: just solve demand allocation

# Matrix form: How much water per zone to meet demand?
# Scenario: Limited water (80,000 mm), need to allocate rationally

A = np.array([
    [20, 0, 0],     # Zone A: 20 ha
    [0, 30, 0],     # Zone B: 30 ha
    [0, 0, 15]      # Zone C: 15 ha
], dtype=float)

b = np.array([1000, 1000, 1000])  # Water depth (mm) per zone

# Solve: A @ x = b => x = water depth per zone
x = gaussian_elimination(A, b)

print(f"\n=== WATER ALLOCATION PROBLEM ===")
print(f"Water allocation (mm per zone): {x}")
print(f"Total water needed: {(x * np.array([20, 30, 15])).sum():.0f} mm-hectares")
print(f"Verification: A @ x = {A @ x}")

In [ ]:
# CONVERGENCE COMPARISON: All 3 root-finding methods

# Problem: Find zero of f(x) = x^2 - 2 (root is sqrt(2) ≈ 1.414)
def f(x):
    return x**2 - 2

def df(x):
    return 2*x

# Bisection
root_bis, iter_bis = bisection(f, 1.0, 2.0, tol=1e-10)

# Newton-Raphson
root_nr, iter_nr = newton_raphson(f, df, 2.0, tol=1e-10)

# Secant
root_sec, iter_sec = secant(f, 1.0, 2.0, tol=1e-10)

exact_root = np.sqrt(2)

comparison = pd.DataFrame({
    'Method': ['Bisection', 'Newton-Raphson', 'Secant'],
    'Root': [root_bis, root_nr, root_sec],
    'Iterations': [iter_bis, iter_nr, iter_sec],
    'Error': [
        abs(root_bis - exact_root),
        abs(root_nr - exact_root),
        abs(root_sec - exact_root)
    ]
})

print(f"\n=== ROOT-FINDING CONVERGENCE (f(x) = x² - 2) ===")
print(f"Exact root (√2): {exact_root:.10f}")
print(f"\n{comparison.to_string(index=False)}")
print(f"\n✓ Newton-Raphson is fastest (quadratic convergence)")
print(f"✓ Bisection is slowest but most robust")
print(f"✓ Secant is middle ground (doesn't require derivative)")

In [ ]:
# SUMMARY: Level 3 Outcomes
print("\n" + "="*60)
print("LEVEL 3 SUMMARY: Numerical Methods")
print("="*60)

print("\n✓ Root-Finding Applications:")
print(f"  • Optimal irrigation: {i_opt:.2f} mm needed to reach target")
print(f"  • Method comparison: NR (fastest), Bisection (robust), Secant (derivative-free)")

print("\n✓ Numerical Integration:")
print(f"  • Cumulative ET over season: {total_et_trap:.2f} mm")
print(f"  • Simpson's rule error: {abs(integral_simp - 2.0):.2e} (very accurate)")

print("\n✓ Linear Algebra:")
print(f"  • Gaussian elimination: Solves Ax=b for water allocation")
print(f"  • Example: {x} mm per zone")

print("\n✓ Ready for Level 4: Data cleaning and visualization")